# Exercise: Multi-Modal Retrieval Augmented Generation from Scratch

## Objective

Implement a Retrieval Augmented Generation (RAG) system from scratch using the [PHI-3 vision](https://huggingface.co/microsoft/Phi-3-vision-128k-instruct) model, [Jina-CLIP-V1](https://huggingface.co/jinaai/jina-clip-v1) embeddings, and the Chroma vector database. Your task is to use this system to generate text based on the contents of the paper "Attention is All You Need" ([https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)).

## Allowed Libraries

You are only permitted to use the following libraries:
- `torch`
- `chromadb`
- `numpy`
- `io`
- `fitz`
- `requests`
- `PIL`
- `transformers`

## Components to Implement

1. **Document Ingestion**: Load the "Attention is All You Need" paper and prepare it for processing. This includes handling the PDF and extracting text and images as needed.

2. **Embeddings Generation**: Use Jina-CLIP-V1 to create embeddings for the text and images extracted from the document.

3. **Vector Database Management**: Utilize the Chroma vector database to store and manage the embeddings.

4. **Retrieval Mechanism**: Implement a mechanism to retrieve relevant document segments (text or images) based on a given query using the stored embeddings.

5. **Generation Model**: Integrate the PHI-3 vision model to generate new content based on the retrieved segments.

## Submission

Your submission should include:
- The complete code for the exercise.
- A README file explaining your implementation and how to run the code.
- Examples of generated outputs based on sample queries.

## Evaluation Criteria

- Correctness: The RAG system should accurately retrieve relevant document segments and generate coherent outputs.
- Efficiency: The implementation should be optimized for performance.
- Creativity: Innovative approaches to integrate and utilize the components are encouraged.
- Clarity: Code should be well-documented and easy to understand.

Good luck and enjoy the exercise!

In [1]:
# !pip install -q "transformers==4.44.2" accelerate einops timm
# !pip install -q chromadb pymupdf pillow requests
# #flash-attn is intentionally NOT installed; both models below use eager attention.
# !pip install fitz
# !pip install tools

In [2]:
!pip uninstall -y fitz frontend PyMuPDF PyMuPDFb
!pip install PyMuPDF
import fitz
print(fitz.__doc__)   # should now show the PyMuPDF banner
print(fitz.__file__)

Found existing installation: fitz 0.0.1.dev2
Uninstalling fitz-0.0.1.dev2:
  Successfully uninstalled fitz-0.0.1.dev2
Found existing installation: pymupdf 1.28.0
Uninstalling pymupdf-1.28.0:
  Successfully uninstalled pymupdf-1.28.0
  Using cached pymupdf-1.28.0-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (26 kB)
Using cached pymupdf-1.28.0-cp310-abi3-manylinux_2_28_x86_64.whl (25.7 MB)
PyMuPDF 1.28.0: Python bindings for the MuPDF 1.29.0 library.
Python 3.12 running on linux (64-bit).

/usr/local/lib/python3.12/dist-packages/fitz/__init__.py


## Section 1: Imports & configuration

Importing the allowed libraries, detects the GPU, and centralises every tunable knob (chunk
size/overlap, image-size filter, model IDs, `top_k`, token budget) in one `CFG` object.

In [3]:
import io
import requests
import numpy as np
import torch
import fitz  # PyMuPDF
from PIL import Image
import chromadb

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: No GPU detected. Phi-3-Vision will be extremely slow / may OOM. "
          "Switch to a GPU runtime.")


class CFG:
    PDF_URL = ("https://proceedings.neurips.cc/paper_files/paper/2017/file/"
               "3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf")

    # --- Text chunking (word-based, with overlap to preserve context across cuts) ---
    CHUNK_SIZE = 220      # words per chunk
    CHUNK_OVERLAP = 40    # words shared between consecutive chunks

    # --- Image filtering (drop logos / tiny artifacts) ---
    MIN_IMG_DIM = 100     # min width AND height in pixels

    # --- Models ---
    JINA_ID = "jinaai/jina-clip-v1"
    PHI_ID = "microsoft/Phi-3-vision-128k-instruct"

    # --- Retrieval ---
    TOP_K = 5             # total nearest neighbours to fetch
    MIN_IMAGES = 1        # always surface at least this many figures (multimodal boost)
    MAX_IMAGES_TO_MODEL = 2  # cap images fed to Phi-3-Vision (memory / context control)

    # --- Generation ---
    MAX_NEW_TOKENS = 512


cfg = CFG()

Torch: 2.11.0+cu128 | Device: cuda


## Section 2: Document ingestion (text + figures)

Streams the PDF into memory with `requests` and opens it with `fitz`. Text is split into
overlapping word windows (keeping each chunk's page number for citations); every sizeable
embedded figure is decoded to a `PIL` image and tiny artifacts are filtered out.

In [4]:
def download_pdf(url: str) -> bytes:
    """Download the PDF into memory (a User-Agent avoids occasional 403s)."""
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    resp.raise_for_status()
    return resp.content


def extract_text_chunks(doc, chunk_size, overlap):
    """Return a list of {'text', 'page'} dicts using an overlapping sliding window."""
    step = max(1, chunk_size - overlap)
    chunks = []
    for page_index in range(len(doc)):
        words = doc[page_index].get_text("text").split()
        if not words:
            continue
        start = 0
        while start < len(words):
            window = words[start:start + chunk_size]
            text = " ".join(window).strip()
            if len(text) > 40:  # skip near-empty fragments
                chunks.append({"text": text, "page": page_index + 1})
            if start + chunk_size >= len(words):
                break
            start += step
    return chunks


def extract_images(doc, min_dim):
    """Return a list of {'image' (PIL), 'page', 'xref'} for every sizeable embedded image."""
    images, seen = [], set()
    for page_index in range(len(doc)):
        for img in doc[page_index].get_images(full=True):
            xref = img[0]
            if xref in seen:            # the same image can appear on several pages
                continue
            seen.add(xref)
            try:
                base = doc.extract_image(xref)
                pil = Image.open(io.BytesIO(base["image"])).convert("RGB")
            except Exception:
                continue
            if pil.width < min_dim or pil.height < min_dim:
                continue
            images.append({"image": pil, "page": page_index + 1, "xref": xref})
    return images


# --- Run ingestion ---
pdf_bytes = download_pdf(cfg.PDF_URL)
doc = fitz.open(stream=pdf_bytes, filetype="pdf")

text_chunks = extract_text_chunks(doc, cfg.CHUNK_SIZE, cfg.CHUNK_OVERLAP)
image_items = extract_images(doc, cfg.MIN_IMG_DIM)

print(f"Pages: {len(doc)}")
print(f"Text chunks: {len(text_chunks)}")
print(f"Figures extracted: {len(image_items)}  "
      f"(sizes: {[im['image'].size for im in image_items]})")
print("\nExample chunk (page {}):\n{}...".format(
    text_chunks[0]['page'], text_chunks[0]['text'][:280]))

Pages: 11
Text chunks: 30
Figures extracted: 3  (sizes: [(1520, 2239), (835, 1282), (445, 884)])

Example chunk (page 1):
Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† Unive...


## Section 3: Embeddings with Jina-CLIP-v1

Loads Jina-CLIP-v1, whose `encode_text` / `encode_image` map **into one shared 768-d space**
(this is what lets a text query retrieve a diagram). Embeddings are L2-normalised for clean
cosine similarity, then the whole corpus is embedded in batches under `no_grad`.

In [5]:
from transformers import AutoModel

jina = AutoModel.from_pretrained(
    cfg.JINA_ID, trust_remote_code=True
).to(DEVICE).eval()


def _l2norm(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


@torch.no_grad()
def embed_texts(texts):
    """texts: list[str] -> (N, 768) float32, L2-normalised."""
    embs = jina.encode_text(texts, batch_size=32)
    return _l2norm(embs)


@torch.no_grad()
def embed_images(pil_images):
    """pil_images: list[PIL.Image] -> (N, 768) float32, L2-normalised."""
    embs = jina.encode_image(pil_images, batch_size=8)
    return _l2norm(embs)


# --- Embed the corpus ---
text_embeddings = embed_texts([c["text"] for c in text_chunks])
image_embeddings = (embed_images([im["image"] for im in image_items])
                    if image_items else np.zeros((0, text_embeddings.shape[1]),
                                                  dtype=np.float32))

print("Text embeddings :", text_embeddings.shape)
print("Image embeddings:", image_embeddings.shape)

# Sanity check: a text query about attention should be closer to text chunks that mention it.
_probe = embed_texts(["scaled dot-product attention"])[0]
_sims = text_embeddings @ _probe
print("Top matching chunk page for probe 'scaled dot-product attention':",
      text_chunks[int(np.argmax(_sims))]["page"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- configuration_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_clip.py: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

processing_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- processing_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Text embeddings : (30, 768)
Image embeddings: (3, 768)
Top matching chunk page for probe 'scaled dot-product attention': 4


## Section 4: Vector database (ChromaDB)

Creates one cosine-distance collection and adds both modalities with our precomputed vectors
(Chroma does **not** re-embed). Text chunks store their text; figures store only metadata
while the actual `PIL` images stay in an in-memory `IMAGE_STORE` keyed by the same id.

In [6]:
client = chromadb.Client()
# Fresh start if the cell is re-run
try:
    client.delete_collection("attention_paper")
except Exception:
    pass

collection = client.get_or_create_collection(
    name="attention_paper",
    metadata={"hnsw:space": "cosine"},
)

IMAGE_STORE = {}  # id -> PIL.Image (kept in RAM; Chroma only holds vectors + metadata)

# --- Add text chunks ---
if text_chunks:
    collection.add(
        ids=[f"text_{i}" for i in range(len(text_chunks))],
        embeddings=text_embeddings.tolist(),
        documents=[c["text"] for c in text_chunks],
        metadatas=[{"type": "text", "page": c["page"]} for c in text_chunks],
    )

# --- Add figures ---
if image_items:
    img_ids = [f"image_{i}" for i in range(len(image_items))]
    for iid, im in zip(img_ids, image_items):
        IMAGE_STORE[iid] = im["image"]
    collection.add(
        ids=img_ids,
        embeddings=image_embeddings.tolist(),
        documents=[f"[Figure located on page {im['page']}]" for im in image_items],
        metadatas=[{"type": "image", "page": im["page"]} for im in image_items],
    )

print("Collection size:", collection.count(),
      f"({len(text_chunks)} text + {len(image_items)} images)")

Collection size: 33 (30 text + 3 images)


## Section 5: Retrieval

Embeds the query with the *same* Jina-CLIP text encoder and pulls the nearest neighbours by
cosine distance. Because text-vs-text similarity usually beats text-vs-image, a metadata
filter guarantees at least one figure is available so diagram questions actually use a figure.

In [7]:
def _unpack(res):
    """Flatten a single-query Chroma response into a list of dicts."""
    items = []
    for _id, doc_, meta, dist in zip(
        res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]
    ):
        items.append({"id": _id, "document": doc_, "metadata": meta, "distance": dist})
    return items


def retrieve(query, top_k=None, min_images=None):
    top_k = top_k or cfg.TOP_K
    min_images = cfg.MIN_IMAGES if min_images is None else min_images

    q_emb = embed_texts([query])[0].tolist()
    items = _unpack(collection.query(query_embeddings=[q_emb], n_results=top_k))

    # Ensure at least `min_images` figures are available for the generator.
    have = sum(1 for it in items if it["metadata"]["type"] == "image")
    if min_images > 0 and have < min_images and len(IMAGE_STORE) > 0:
        extra = _unpack(collection.query(
            query_embeddings=[q_emb],
            n_results=min_images,
            where={"type": "image"},
        ))
        present = {it["id"] for it in items}
        items += [e for e in extra if e["id"] not in present]

    return items


# --- Smoke test ---
for it in retrieve("How does multi-head attention work?"):
    print(f"{it['id']:>10} | page {it['metadata']['page']:>2} | "
          f"{it['metadata']['type']:>5} | dist={it['distance']:.3f}")

   text_10 | page  5 |  text | dist=0.246
    text_8 | page  4 |  text | dist=0.384
    text_9 | page  4 |  text | dist=0.440
    text_7 | page  3 |  text | dist=0.470
    text_4 | page  2 |  text | dist=0.529
   image_0 | page  3 | image | dist=0.766


## Section 6: Generation with Phi-3-Vision

Loads Phi-3-Vision (eager attention, no flash-attn) and builds a prompt from the retrieved
text plus figures, using `<|image_i|>` placeholders whose count must equal the images passed.
The model is told to answer **only** from the context and cite pages — this keeps it grounded.

In [8]:
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download(
    repo_id="microsoft/Phi-3-vision-128k-instruct",
    local_dir="./Phi-3-vision-128k-instruct",
    local_dir_use_symlinks=False,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

configuration_phi3_v.py: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

processing_phi3_v.py: 0.00B [00:00, ?B/s]

sample_inference.py: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

In [9]:
print("MODEL_DIR:", MODEL_DIR)

MODEL_DIR: /content/Phi-3-vision-128k-instruct


In [10]:
import os
import subprocess
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

# Local directory where the model will be stored
MODEL_DIR = "./Phi-3-vision-128k-instruct"

# Download once if not already present
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("Downloading Phi-3 Vision model (first time only)...")
    subprocess.run([
        "huggingface-cli",
        "download",
        "microsoft/Phi-3-vision-128k-instruct",
        "--local-dir",
        MODEL_DIR,
    ], check=True)
    print("Download complete.")
else:
    print("Using cached local model.")

# Load model from local directory
phi_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    device_map="auto" if DEVICE == "cuda" else None,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    _attn_implementation="eager",
).eval()

phi_processor = AutoProcessor.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    num_crops=4,
)


def generate_answer(query, retrieved, max_new_tokens=None):
    max_new_tokens = max_new_tokens or cfg.MAX_NEW_TOKENS

    text_ctx = [r for r in retrieved if r["metadata"]["type"] == "text"]
    img_ctx = [r for r in retrieved if r["metadata"]["type"] == "image"][
        : cfg.MAX_IMAGES_TO_MODEL
    ]
    images = [IMAGE_STORE[r["id"]] for r in img_ctx]

    context_block = "\n\n".join(
        f"[Excerpt — page {r['metadata']['page']}]\n{r['document']}"
        for r in text_ctx
    ) or "(no text excerpts retrieved)"

    placeholders = "".join(
        f"<|image_{i + 1}|>\n" for i in range(len(images))
    )

    if images:
        placeholders = (
            "The following figure(s) were retrieved from the paper:\n"
            + placeholders
            + "\n"
        )

    user_content = (
        f"{placeholders}"
        "You are a precise research assistant. Using ONLY the context below from "
        "the paper 'Attention Is All You Need', answer the question. If the "
        "context is insufficient, say so. Cite page numbers where relevant.\n\n"
        f"=== CONTEXT ===\n{context_block}\n\n"
        f"=== QUESTION ===\n{query}"
    )

    messages = [{"role": "user", "content": user_content}]

    prompt = phi_processor.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = phi_processor(
        prompt,
        images if images else None,
        return_tensors="pt",
    ).to(phi_model.device)

    with torch.no_grad():
        outputs = phi_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=phi_processor.tokenizer.eos_token_id,
        )

    outputs = outputs[:, inputs["input_ids"].shape[1]:]

    answer = phi_processor.batch_decode(
        outputs,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return answer.strip(), img_ctx

Using cached local model.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


## Section 7: End-to-end RAG pipeline

`rag()` ties retrieval and generation together and prints its provenance (which pages and
figures fed the answer) so every result stays auditable. This is the single function you call
for any question.

In [11]:
def rag(query, top_k=None, max_new_tokens=None, verbose=True):
    retrieved = retrieve(query, top_k=top_k)
    if verbose:
        print(f"Q: {query}")
        print("Retrieved context:")
        for r in retrieved:
            print(f"   - {r['id']} (page {r['metadata']['page']}, "
                  f"{r['metadata']['type']}, dist={r['distance']:.3f})")
    answer, used_images = generate_answer(query, retrieved, max_new_tokens)
    if verbose:
        if used_images:
            print("Figures shown to the model:",
                  [f"{u['id']} (p{u['metadata']['page']})" for u in used_images])
        print("\nAnswer:\n" + answer + "\n" + "-" * 80)
    return answer

## Section 8: Example queries

Four sample questions exercising the full path. The first three are text-centric; the last is
figure-centric and should pull in the architecture diagram.

**8.1: Transformer vs. recurrent / convolutional models**

In [13]:
_ = rag("What is the Transformer and how does it differ from recurrent and "
        "convolutional sequence models?")

Q: What is the Transformer and how does it differ from recurrent and convolutional sequence models?
Retrieved context:
   - text_2 (page 2, text, dist=0.321)
   - text_5 (page 2, text, dist=0.396)
   - text_1 (page 1, text, dist=0.406)
   - text_3 (page 2, text, dist=0.417)
   - text_4 (page 2, text, dist=0.426)
   - image_1 (page 4, image, dist=0.889)


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
/root/.cache/huggingface/modules/transformers_modules/Phi-3-vision-128k-instruct/image_embedding_phi3_v.py:197: UserWarning: Phi-3-V modifies `input_ids` in-place and the tokens indicating images will be removed after model forward. If your workflow requires multiple forward passes on the same `input_ids`, please make a copy of `input_ids` before passing it to the model.
  warnings.warn(


Figures shown to the model: ['image_1 (p4)']

Answer:
The Transformer is a model architecture that eschews recurrence and instead relies entirely on an attention mechanism to draw global dependencies between input and output. It differs from recurrent and convolutional sequence models in that it allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs. Unlike recurrent models, the Transformer does not factor computation along the symbol positions of the input and output sequences, which precludes parallelization within training examples. This is a fundamental constraint of sequential computation, which is inherent in recurrent models. In contrast, the Transformer uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, allowing it to model dependencies without regard to their distance in the input or output sequences. Addition

**8.2: Multi-head attention and its benefits**

In [14]:
_ = rag("Explain multi-head attention and why it is beneficial compared to a single "
        "attention head.")

Q: Explain multi-head attention and why it is beneficial compared to a single attention head.
Retrieved context:
   - text_10 (page 5, text, dist=0.303)
   - text_8 (page 4, text, dist=0.430)
   - text_9 (page 4, text, dist=0.487)
   - text_7 (page 3, text, dist=0.510)
   - text_3 (page 2, text, dist=0.572)
   - image_0 (page 3, image, dist=0.796)
Figures shown to the model: ['image_0 (p3)']

Answer:
Multi-head attention is a mechanism that allows the model to jointly attend to information from different representation subspaces at different positions. It does this by performing the attention function in parallel on several versions of queries, keys, and values, each projected with different learned linear projections. This is beneficial compared to a single attention head because it allows for more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs. Additionally, multi-head attention can coun

**8.3: Translation results (BLEU) and training cost**

In [15]:
_ = rag("What BLEU scores did the Transformer achieve on English-to-German and "
        "English-to-French translation, and how did its training cost compare?")

Q: What BLEU scores did the Transformer achieve on English-to-German and English-to-French translation, and how did its training cost compare?
Retrieved context:
   - text_19 (page 8, text, dist=0.155)
   - text_20 (page 8, text, dist=0.292)
   - text_22 (page 9, text, dist=0.410)
   - text_21 (page 8, text, dist=0.477)
   - text_17 (page 7, text, dist=0.523)
   - image_1 (page 4, image, dist=0.991)
Figures shown to the model: ['image_1 (p4)']

Answer:
The Transformer achieved BLEU scores of 27.3 on English-to-German translation and 41.0 on English-to-French translation. Its training cost was significantly lower compared to previous state-of-the-art models, achieving these results at a fraction of the training cost.
--------------------------------------------------------------------------------


**8.4: Architecture diagram (figure-centric)**: this one feeds a retrieved figure to
Phi-3-Vision, so the model *looks at* the diagram while answering.

In [16]:
_ = rag("Describe the overall model architecture shown in the figure, including the "
        "encoder and decoder stacks.")

Q: Describe the overall model architecture shown in the figure, including the encoder and decoder stacks.
Retrieved context:
   - text_5 (page 2, text, dist=0.367)
   - text_6 (page 3, text, dist=0.479)
   - text_12 (page 5, text, dist=0.585)
   - text_11 (page 5, text, dist=0.606)
   - text_22 (page 9, text, dist=0.637)
   - image_1 (page 4, image, dist=0.801)
Figures shown to the model: ['image_1 (p4)']

Answer:
The overall model architecture shown in the figure consists of an encoder and a decoder stack. The encoder is composed of a stack of N = 6 identical layers, each with two sub-layers. The first sub-layer is a multi-head self-attention mechanism, and the second is a simple, position-wise, fully connected layer. The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, the decode